# Practical 10: Logs You Can Actually Search

**Levels, structured JSON events, and what must never be recorded**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L21 |
| **Course Outcome** | CO5 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 350 MB |
| **Extra software** | nothing beyond the course environment |
| **Marks** | 10 |

## Aim

1. State why `print()` is insufficient once software is deployed.
2. Use logging levels to raise and lower detail without editing code.
3. Write one JSON object per event, so logs can be searched by machine.
4. Recognise what must never be written into a log, and what to record instead.

## What this practical is about

A `print` statement is adequate while a person is watching. Deployed software
is not watched. It is read afterwards, by somebody who was not present, often
to answer a question nobody anticipated.

Four properties separate a log record from a printed line.

- A **timestamp**, so events can be ordered and correlated with an incident.
- A **level**, so severity is stated rather than inferred from wording.
- **Structure**, so a field can be queried rather than parsed out of a
  sentence.
- A **destination that is not the code**, so the volume of detail can be
  changed without a rebuild.

One further point is established at the end. A log is retained for months and
read by many people, which makes it the wrong place for anything identifying an
individual. The remedy is not to omit the information but to record a stable
reference to it instead of the value.

## Before you start

P07 should be complete, with a service exposing a `/predict` endpoint.
Familiarity with a Python dictionary and with JSON is assumed.

Run the cells from top to bottom. A later cell normally depends on an object
created by an earlier one, so a `NameError` usually indicates a skipped
prerequisite.

If notebook state becomes confusing, restart the kernel, clear all outputs, and
run again from the first cell.

In [ ]:
# Step 0: check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'joblib']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'joblib']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

## Step 0b: ensure the shared dataset exists

Every practical in this course uses the same 600 synthetic delivery
observations. The next cell confirms the file is present and reconstructs it
from the course seed if it is absent.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
from pathlib import Path

def find_data(name="delivery_times.csv"):
    """Locate the shared dataset from any working directory.

    Walks upward from the current folder looking for data/<name>, so the
    notebook works whether it is launched from the practical folder, the
    Practicals folder, or the repository root."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in (folder / "data" / name,
                          folder / "Practicals" / "data" / name):
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError(
        f"{name} not found in any data/ folder at or above {here}")

DATA = find_data()
print("dataset:", DATA)


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

# Walkthrough

Read each step, then run its cell.

## Step 1: observe what print omits

The cell below prints a line. Read what it provides and, more importantly, what
it does not.

In [ ]:
print("something went wrong")

There is no time, so the line cannot be placed against an incident. There is no
severity, so whether this is routine or fatal must be inferred from the
wording. There is no indication of which request it belongs to. And there is no
way to suppress it in production without editing the code.

Each of those is addressed by the logging module, and none of them is addressed
by writing a longer message.

## Step 2: configure a logger

`logging.getLogger(name)` returns a named logger. Its output is directed to the
screen with a `StreamHandler`, and the format is decided once rather than at
every call site.

The `handlers.clear()` line matters specifically in notebooks. Without it, each
re-run of the cell attaches a further handler, and every message then appears
twice, then three times.

In [ ]:
import logging
import sys
from pathlib import Path

WORK = Path("work")
WORK.mkdir(exist_ok=True)

logger = logging.getLogger("delivery")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()          # re-running this cell is now safe
logger.propagate = False

to_screen = logging.StreamHandler(sys.stdout)
to_screen.setFormatter(logging.Formatter(
    "%(asctime)s  %(levelname)-8s  %(name)s  %(message)s"
))
logger.addHandler(to_screen)

logger.info("logger ready")

## Step 3: the five levels

Run the cell and read the output. Every line now carries a time and a stated
severity.

In [ ]:
logger.debug("feature vector built: [5.0, 20, 2, 0]")
logger.info("prediction served in 3 ms")
logger.warning("distance 47 km is unusually far")
logger.error("model file could not be read")
logger.critical("out of memory, shutting down")

Choosing a level is a judgement made many times. One test is serviceable: would
this justify waking somebody? If so, `ERROR` or `CRITICAL`. If it is useful
history rather than a demand for attention, `INFO`.

## Step 4: reduce the detail without touching the code

This is the purpose of levels. Changing one threshold removes the `DEBUG` and
`INFO` lines everywhere at once.

In [ ]:
logger.setLevel(logging.WARNING)

print("--- threshold is now WARNING ---")
logger.debug("you will not see this")
logger.info("nor this")
logger.warning("but you will see this")
logger.error("and this")

logger.setLevel(logging.DEBUG)     # back to chatty for the lab

Production normally runs at `INFO`. When something is wrong the threshold is
lowered to `DEBUG` for a short period and then raised again. No rebuild and no
redeployment is involved, which is why the threshold is a setting and not a
constant in the code.

## Step 5: one line, one event, as JSON

The lines are now made machine readable. A **formatter** determines what a log
line looks like, and the one below produces a JSON object rather than a
sentence.

Any additional field the caller supplies appears in the same object. That is
the property which makes the logs answerable later.

In [ ]:
import json
from datetime import datetime, timezone


class JsonFormatter(logging.Formatter):
    """Turn one log record into one line of JSON."""

    def format(self, record):
        event = {
            "time": datetime.fromtimestamp(
                record.created, timezone.utc
            ).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        # anything passed as extra={...} is added here
        for key, value in getattr(record, "fields", {}).items():
            event[key] = value
        return json.dumps(event)


def log_event(level, message, **fields):
    """Log one message with named fields attached."""
    logger.log(level, message, extra={"fields": fields})


to_screen.setFormatter(JsonFormatter())

log_event(logging.INFO, "prediction served",
          duration_ms=3, distance_km=5.0, predicted_min=42.1)

Read that line. Every fact carries a name. A search tool can now answer a
question such as the mean `duration_ms` yesterday without anyone writing a
parser for the message text, and the question did not have to be anticipated
when the line was written.

## Step 6: send the same events to a file as well

On a laptop a copy on disk is convenient, and Step 8 searches it. Inside a
container this would not be done. There the service writes to standard output
only and the platform collects it, which is the arrangement L21 measured:
`docker logs` shows the stream, while a file inside the container is reachable
only through `docker exec`.

The file format here is JSONL: one JSON object per line.

In [ ]:
LOG_FILE = WORK / "service.log"
LOG_FILE.unlink(missing_ok=True)

to_file = logging.FileHandler(LOG_FILE, encoding="utf-8")
to_file.setFormatter(JsonFormatter())
logger.addHandler(to_file)

log_event(logging.INFO, "service started", version="1.0.0")
print("logging to", LOG_FILE)

## Step 7: log a service doing its work

The cell below simulates 40 predictions and logs each one: what arrived, what
was returned, and how long it took. Two of them fail deliberately.

Note `request_id`. When a problem is reported hours afterwards, that identifier
is how the relevant line is located among thousands.

In [ ]:
import random
import time
import uuid

import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
orders = pd.read_csv(DATA)
X_train, X_test, y_train, y_test = train_test_split(
    orders[FEATURES], orders["delivery_min"],
    test_size=0.2, random_state=42,
)
model = LinearRegression().fit(X_train, y_train)

rng = random.Random(42)
to_screen.setLevel(logging.WARNING)   # keep the notebook readable

for i in range(40):
    request_id = str(uuid.uuid4())[:8]
    row = X_test.iloc[[i]]
    started = time.perf_counter()

    if i in (11, 27):                 # two deliberate failures
        log_event(logging.ERROR, "prediction failed",
                  request_id=request_id,
                  error="ValueError: traffic_level out of range",
                  duration_ms=round(
                      (time.perf_counter() - started) * 1000, 2))
        continue

    minutes = float(model.predict(row)[0])
    duration = (time.perf_counter() - started) * 1000
    if rng.random() < 0.15:           # a few slow ones
        duration += rng.uniform(200, 400)

    log_event(logging.INFO, "prediction served",
              request_id=request_id,
              distance_km=float(row["distance_km"].iloc[0]),
              traffic_level=int(row["traffic_level"].iloc[0]),
              predicted_min=round(minutes, 1),
              duration_ms=round(duration, 2))

to_screen.setLevel(logging.NOTSET)
print("40 requests logged to", LOG_FILE)
print("lines written:",
      len(LOG_FILE.read_text(encoding='utf-8').splitlines()))

Only the two errors appeared on screen, because the screen handler's threshold
was raised to `WARNING`. Everything still reached the file. Two destinations,
two thresholds, one line of configuration each.

## Step 8: search the logs as a machine would

This is the reason for the JSON. The file is read back and real questions are
answered with ordinary Python, with no parsing of prose.

In [ ]:
events = [json.loads(line)
          for line in LOG_FILE.read_text(encoding="utf-8").splitlines()
          if line.strip()]

print("total events :", len(events))

errors = [e for e in events if e["level"] == "ERROR"]
print("errors       :", len(errors))

served = [e for e in events if e.get("message") == "prediction served"]
durations = [e["duration_ms"] for e in served]
print("predictions  :", len(served))
print("average ms   :", round(sum(durations) / len(durations), 1))

slow = [e for e in served if e["duration_ms"] > 200]
print("slower than 200 ms:", len(slow))
print()
print("The slowest request:")
worst = max(served, key=lambda e: e["duration_ms"])
print(f"  request_id {worst['request_id']}  "
      f"{worst['duration_ms']} ms  "
      f"{worst['distance_km']} km")

Each of those questions would otherwise have required reading sentences by eye.
That difference is what structure buys, and it is why the decision to log JSON
is made before the questions are known.

## Step 9: what must never appear in a log

Logs are retained for months and read by many people. The following must never
be written into one.

- Names, telephone numbers and exact home addresses.
- Card numbers, passwords, API keys and tokens.
- Anything that identifies one individual customer.

The value is almost never what is needed. What is needed is the ability to
match. A one way hash records that two orders came from the same customer
without recording who that customer is.

In [ ]:
import hashlib

bad_order = {
    "customer_name": "Ananya Sharma",
    "phone": "9876543210",
    "address": "Flat 402, Green Park, Noida",
    "distance_km": 5.0,
    "traffic_level": 2,
}


def safe_for_logging(order):
    """Keep what helps debugging, remove what identifies a person."""
    secret = str(order.get("phone", ""))
    return {
        "customer_ref": hashlib.sha256(
            secret.encode()).hexdigest()[:12],
        "distance_km": order["distance_km"],
        "traffic_level": order["traffic_level"],
    }


print("NEVER log this:")
print("  ", bad_order)
print()
print("Log this instead:")
print("  ", safe_for_logging(bad_order))

log_event(logging.INFO, "order received",
          **safe_for_logging(bad_order))

`customer_ref` is stable: the same telephone number always yields the same
reference, so a customer's orders can still be grouped. Nobody reading the log
learns a telephone number. This is the alternative L29 asks for when a teammate
proposes logging the number itself.

## Step 10: where logs go once a cluster is involved

The operational picture from L21. None of it needs to be run today. It needs to
be drawable.

In [ ]:
pipeline = [
    ("your code", "writes one JSON line per event to stdout"),
    ("the container runtime", "captures stdout to a file on the node"),
    ("a collector (Fluent Bit / FluentD)",
     "runs on every node and reads those files"),
    ("a store (Elasticsearch / OpenSearch)",
     "indexes the events so they can be searched"),
    ("a viewer (Kibana / Grafana)", "is where a human asks questions"),
]

for i, (part, does) in enumerate(pipeline, 1):
    print(f"  {i}. {part}")
    print(f"     {does}")

print()
print("Two rules that follow from this picture:")
print("  - never write logs to a file inside a container;")
print("    the container dies and takes the file with it")
print("  - never log secrets; that store is searchable by")
print("    everyone in the company")

# Your turn

The walkthrough is complete. The remaining work is yours.

There are **three tasks**, in increasing order of difficulty, each with a hint.

Where a `# TODO` comment appears, replace that line with your own code. Do not
rename the variable on the left of the `=` sign. The self-check at the end
looks for exactly those names.

When all three have been attempted, run the **self-check** cell. It may be run
as often as required and it never alters the notebook.

## Task T1: decide what is recorded

A colleague requires the service to record warnings and errors only, and
nothing routine.

Set the `logger` threshold so that `DEBUG` and `INFO` are discarded while
`WARNING`, `ERROR` and `CRITICAL` are retained. Then build a list `T1_kept`
naming the levels that would still be recorded, as uppercase strings.

> **Hint.** `logger.setLevel(logging.WARNING)` sets the threshold. For the list,
> ask the logger itself with `logger.isEnabledFor(logging.DEBUG)` and so on
> rather than writing the answer from memory. The point of the task is that the
> code states the behaviour instead of asserting it.

In [ ]:
LEVELS = {"DEBUG": logging.DEBUG, "INFO": logging.INFO,
          "WARNING": logging.WARNING, "ERROR": logging.ERROR,
          "CRITICAL": logging.CRITICAL}

# TODO: set the logger threshold so only WARNING and above survive

# TODO: which level names would still be recorded?
T1_kept = None

print("still recorded:", T1_kept)

## Task T2: a formatter for the delivery service

Write a class `ServiceFormatter`, deriving from `logging.Formatter`, whose
`format` method returns one line of JSON with exactly these five keys.

- `"ts"`, the time as an ISO string.
- `"level"`, the level name.
- `"service"`, always the string `"delivery-api"`.
- `"msg"`, the message.
- `"env"`, always the string `"lab"`.

Then create one instance, call `format` on a test record, and store the parsed
dictionary in `T2_line`.

> **Hint.** Follow `JsonFormatter` from Step 5 and change the keys. The stub
> provides `sample_record` to test with. Call
> `ServiceFormatter().format(sample_record)` and parse the result with
> `json.loads`.

In [ ]:
sample_record = logger.makeRecord(
    "delivery", logging.INFO, "x", 1, "prediction served", None, None
)


class ServiceFormatter(logging.Formatter):
    def format(self, record):
        # TODO: return one line of JSON with the five required keys
        return "{}"


# TODO: format the sample record and parse it back into a dictionary
T2_line = None

print(T2_line)

## Task T3: answer a question from the logs

A manager asks what fraction of the predictions served took longer than 100
milliseconds, and which was the slowest.

Using the `events` list from Step 8, determine

- `T3_slow_fraction`, the fraction between 0 and 1 of `"prediction served"`
  events whose `duration_ms` exceeds 100,
- `T3_worst_id`, the `request_id` of the slowest.

> **Hint.** Reduce `events` to those whose `message` is `"prediction served"`.
> The fraction is the length of the slow subset divided by the length of the
> served subset. For the slowest, apply `max(served, key=...)` on `duration_ms`
> and read its `request_id`.

In [ ]:
served_events = [e for e in events
                 if e.get("message") == "prediction served"]

# TODO: what fraction took longer than 100 ms?
T3_slow_fraction = None

# TODO: the request_id of the slowest one
T3_worst_id = None

print("fraction over 100 ms:", T3_slow_fraction)
print("slowest request     :", T3_worst_id)

## Self-check

Run the cell below to mark the work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK: run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | the logger threshold is WARNING', lambda: logger.level == logging.WARNING)
_check('T1 | T1_kept names exactly the three surviving levels', lambda: set(T1_kept) == {'WARNING', 'ERROR', 'CRITICAL'})
_check('T2 | the line has exactly the five required keys', lambda: set(T2_line) == {'ts', 'level', 'service', 'msg', 'env'})
_check('T2 | service and env carry the fixed values', lambda: T2_line['service'] == 'delivery-api' and T2_line['env'] == 'lab')
_check('T2 | level and message come from the record', lambda: T2_line['level'] == 'INFO' and T2_line['msg'] == 'prediction served')
_check('T2 | it still works for a different record', lambda: (lambda d: d['level'] == 'ERROR' and d['msg'] == 'boom')(json.loads(ServiceFormatter().format(logger.makeRecord('delivery', logging.ERROR, 'x', 1, 'boom', None, None)))))
_check('T3 | T3_slow_fraction matches the log file', lambda: abs(float(T3_slow_fraction) - (len([e for e in events if e.get('message') == 'prediction served' and e['duration_ms'] > 100]) / len([e for e in events if e.get('message') == 'prediction served']))) < 1e-9)
_check('T3 | T3_worst_id is the slowest served request', lambda: T3_worst_id == max([e for e in events if e.get('message') == 'prediction served'], key=lambda e: e['duration_ms'])['request_id'])

print("==================================================================")
print("SELF-CHECK   Practical 10: Logs You Can Actually Search")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

## What to submit

1. This notebook, with every cell run and its output visible.
2. The file `work/service.log`, holding at least 40 JSON lines.
3. In a markdown cell, three fields you would add to every log line in a real
   delivery service, and what question each one makes answerable.

Name the file `P10_<your-roll-number>.ipynb` before uploading it.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, log file produced | 3 |
| Task T1, levels used to control what is recorded | 2 |
| Task T2, a formatter carrying the required fields | 3 |
| Task T3, a real question answered from the logs | 2 |
| **Total** | **10** |

## Read more

- Python, logging HOWTO: <https://docs.python.org/3/howto/logging.html>
- Python, logging cookbook: <https://docs.python.org/3/howto/logging-cookbook.html>
- Twelve factor app, logs as event streams: <https://12factor.net/logs>
- OWASP, logging cheat sheet: <https://cheatsheetseries.owasp.org/cheatsheets/Logging_Cheat_Sheet.html>